Which price? Comparing the sources GLSS7 actually has
=====================================================

**Author:** Ethan Ligon



## What this is



Valuing food you did not buy means choosing a price, and the textbook
presentation offers a menu: the farmgate price the household would receive,
the market price it would pay, and a marketing margin of 30 to 50% between
them.

Session 2 claimed that the question GLSS7 uses to elicit the farmgate price
does not measure one: households answer it with the market price.  This
notebook is where you check that, and it is worth checking, because the
claim is partly about what the survey contains and partly about what
households were thinking when they answered.

-   **Prerequisites:** `lsms_library` on the release kernel, GhanaLSS microdata.



## Setup



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
WAVE = '2016-17'
acq = ghana.food_acquired().xs(WAVE, level='t')

## 1.  How many sources are there really?



In [1]:
fp = ghana.food_prices()
print("food_prices sources:", fp.index.get_level_values('s').unique().tolist())
print("food_acquired sources:", acq.index.get_level_values('s').unique().tolist())
print("community_prices index:", ghana.community_prices().index.names)

**Exercise 1.1.** The list in the lecture names four sources; this survey has
three.  Say which two of the names denote the same instrument.  Then open
[Section 8H](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Section8H-own-produce.pdf) (PDF, opens in a
new tab) and read question 9.
  Which of the four names does the own-production price actually
answer to?  Write down what you expect its relationship to the purchased
unit value to be *before* you compute anything in section 3.



## 1.  Build the three, on a common grain



Prices are only comparable within an item and a unit, and the cluster is the
finest level at which everyone plausibly faces the same price.



In [1]:
pu = acq.xs('purchased', level='s')
pr = acq.xs('produced',  level='s')

unit_value = ((pu.Expenditure / pu.Quantity)
              .replace([np.inf, -np.inf], np.nan).dropna()
              .groupby(['v', 'j', 'u']).median().rename('unit_value'))
own_report = pr.Price.groupby(['v', 'j', 'u']).median().rename('own_report')

cp = ghana.community_prices().xs(WAVE, level='t')
community = ((cp.Price / cp.NumberOfUnits.replace(0, np.nan))
             .groupby(['v', 'j', 'u']).median().rename('community'))

d = pd.concat([unit_value, own_report, community], axis=1)
d.notna().sum()

In [1]:
for pair in [('unit_value', 'own_report'), ('unit_value', 'community'),
             ('own_report', 'community')]:
    print(f"{pair[0]:11s} & {pair[1]:11s}: {len(d[list(pair)].dropna()):>6,} cells")

**Exercise 2.1.** The community table and the own-production prices overlap on
88 cells in the whole survey.  Before going further, decide what you are
willing to conclude from 88 cells, and write it down so you can be held to
it later.



## 1.  Levels and logs



A constant markup, or an iceberg transport cost, is a constant displacement
in logs.  So look in logs.



In [1]:
def logratio(d, num, den):
    x = d[[num, den]].dropna()
    x = x[(x[num] > 0) & (x[den] > 0)]
    return np.log(x[num] / x[den])

for num, den in [('own_report', 'unit_value'), ('community', 'unit_value'),
                 ('community', 'own_report')]:
    lr = logratio(d, num, den)
    print(f"log({num}/{den}): n={len(lr):>6,} median={lr.median():+.3f} "
          f"sd={lr.std():.3f} IQR=[{lr.quantile(.25):+.2f},{lr.quantile(.75):+.2f}]")

Every median comes back at 0.000, which looks like three sources agreeing.
Before believing that, find out how much of it is arithmetic.



In [1]:
r = (d[['own_report', 'unit_value']].dropna()
       .pipe(lambda x: x[(x > 0).all(axis=1)]))
ratio = r.own_report / r.unit_value
prices = pr.Price.dropna()
print(f"exactly 1.00: {100 * np.isclose(ratio, 1).mean():.1f}%")
print(f"obs per cell, produced: median {pr.groupby(['v', 'j', 'u']).size().median():.0f}")
print(f"own-production prices that are whole cedis: {100 * (prices % 1 == 0).mean():.1f}%")
print(f"...and in 1, 2, 5, 10 or 20: {100 * prices.isin([1, 2, 5, 10, 20]).mean():.1f}%")

**Exercise 3.1.** A third of cells match exactly.  Given three observations to
a cell and prices that are overwhelmingly round, work out roughly how often
two independent draws from the same price distribution would tie, and say
whether 33% is more agreement than chance would produce.  This is the
difference between "the sources agree" and "the data are too coarse to
disagree".

**Exercise 3.2.** Redo the log-ratio table after dropping the exact ties.  Does
any pair show a displacement then?  Report what you find either way; a null
is a result here.



## 1.  Which direction is the gap?



This is the part that changes what you would do.



In [1]:
print(ratio.quantile([.1, .25, .5, .75, .9]).round(2).to_string())
print(f"above 1.5: {100 * (ratio > 1.5).mean():.1f}%"
      f"   below 0.5: {100 * (ratio < 0.5).mean():.1f}%")

**Exercise 4.1.** A marketing margin says the household receives *less* than
the market price, and question 9 asks exactly what it would receive, so
own-report should sit below unit value.  At the median it sits exactly on
it.  Session 2's aside reads that as respondents answering with the market
price they can see.  Two other readings are on offer: a genuinely small
margin, and a unit-and-quality mismatch between the two measures.  Say what
pattern in the table above would have been evidence for each, and whether
you see it.  Then keep only cells with at least ten observations on each
side and redo section 3: how much of the spread was thin cells?

**Exercise 4.2.** Question 9 is hypothetical: it asks what a unit *would*
fetch, not what anything sold for.  Uganda's survey records actual crop
sales, and there the valuation comes out 1.23 times the sale price while
buyers pay 1.60 times it.  GLSS7's Section 8 Part C records harvest
disposal, including sales, but the library does not carry it.  If you have
the raw microdata, build a realised farmgate price from it, compare it
against the question-9 answer for the same household and item, and say
whether Ghana looks like Uganda.  If you do not, say what you expect and
why.



## 1.  Does the choice matter?



In [1]:
value = acq.Expenditure.where(acq.Expenditure.notna(), acq.Quantity * acq.Price)
by_source = value.groupby('s').sum()
print((100 * by_source / by_source.sum()).round(1).to_string())

**Exercise 5.1.** Own production is around 29% of food value on the survey's
own prices.  Recompute it valuing own production at the cluster median
*purchased* unit value instead, falling back up the hierarchy when a cell is
empty.  How far does the 29% move, and does the weighted poverty headcount
move with it?

**Exercise 5.2.** Now impose the margin by fiat: value own production at 70%
of the market unit value, as the textbook's 30% would have it, or at 1/1.6
of it, as Uganda's measured wedge would.  Report the own-production share and the headcount.  You have now
produced three defensible numbers for the same country and year.  Say which
you would publish, and what you would have to believe for the other two to
be wrong.



## Exercises



1.  Everything here is GLSS7.  Repeat section 1 for two other countries and
    report which price sources each actually carries.  The lesson session 2
    wants is that the menu of prices is survey-specific, and one country
    cannot establish that.
2.  The unit value is contaminated by quality choice: richer households buy
    better rice and record a higher unit value for "rice".  Session 3 strips
    that out with cluster fixed effects.  Anticipate it here by regressing
    log unit value on log household food expenditure within cluster-item-unit
    cells, and say how much of the dispersion in section 3 that explains.

